# YOLO Instance Segmentation for Door Anatomy
This notebook contains the complete setup and training pipeline to train a YOLO Instance Segmentation model (supporting YOLOv8 / YOLOv11) on the door anatomy dataset.

### Recommended Runtime
Go to **Runtime** > **Change runtime type** and select **GPU** (T4, L4, or A100) to speed up training.

## 1. Install Dependencies

In [ ]:
# Install the latest ultralytics package
!pip install ultralytics

import torch
import ultralytics
ultralytics.checks()

## 2. Prepare Dataset
Upload `yolo_dataset.zip` using the file explorer sidebar on the left, or mount your Google Drive if the dataset is stored there.

In [ ]:
# Mount Google Drive (Optional - uncomment if your zip is in Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Copy the zip file from Drive if needed
# !cp /content/drive/MyDrive/yolo_dataset.zip .

# Unzip dataset
!unzip -q yolo_dataset.zip -d /content/yolo_dataset

# Verify that dataset.yaml is present
!ls /content/yolo_dataset

## 3. Train YOLO Instance Segmentation Model
We can train using either **YOLOv8** or the latest **YOLOv11** segmentation model.

In [ ]:
from ultralytics import YOLO

# Option A: Train latest YOLO11 Segment model (Recommended)
model_name = 'yolo11n-seg.pt'

# Option B: Train YOLOv8 Segment model
# model_name = 'yolov8n-seg.pt'

print(f"Initializing {model_name}...")
model = YOLO(model_name)

# Train model on GPU
results = model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # GPU device 0
    project='yolo_door_segmentation',
    name='train_run',
    save=True,
    plots=True
)

## 4. Evaluate & Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt
import cv2
import glob
import os

# 1. Find validation prediction images generated during training
val_images = glob.glob('yolo_door_segmentation/train_run/val_batch0_labels.jpg')
if val_images:
    img = cv2.imread(val_images[0])
    plt.figure(figsize=(12, 12))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Ground Truth Validation Batch")
    plt.show()

# 2. Run inference on a few validation floorplans using the best weights
best_weights = 'yolo_door_segmentation/train_run/weights/best.pt'
if os.path.exists(best_weights):
    trained_model = YOLO(best_weights)
    
    # Get 3 validation images
    val_img_paths = glob.glob('/content/yolo_dataset/images/val/*.png')[:3]
    
    for path in val_img_paths:
        results = trained_model.predict(source=path, imgsz=640, conf=0.25)
        
        # Plot and show predicted mask
        for res in results:
            plotted = res.plot()
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.title(f"Prediction: {os.path.basename(path)}")
            plt.show()
else:
    print("Trained weights not found. Please complete training first!")

## 5. Export Best Weights
Run this cell to download the best weights to your local machine.

In [ ]:
from google.colab import files
import os

best_weights = 'yolo_door_segmentation/train_run/weights/best.pt'
if os.path.exists(best_weights):
    files.download(best_weights)
else:
    print("Trained weights not found.")